# Character Appearance Agent 테스트 (Production Level)

캐릭터 외형 추출 에이전트 테스트 노트북

## 역할: "Visual Designer" (비주얼 디자이너)
- 체형 (physique)
- 머리 (hair_style, hair_color)
- 눈, 코, 입
- 복장 (attire)
- 흉터, 문신 (scars_tattoos)

## Production Features (v2.0)
- ✅ **Null Fallback**: null → "unspecified" 또는 role-based defaults
- ✅ **Color Normalization**: natural language → hex code + category
- ✅ **Prompt Aggregation**: full_visual_prompt 자동 생성
- ✅ **Style Context**: art_style, rendering_engine 메타데이터

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
SAMPLE_STORY = """아린은 어두운 숲 한가운데 서 있었다. 스물다섯 살의 젊은 여전사는 긴 검은 머리카락을 바람에 휘날리며, 손에 쥔 은빛 검을 꼭 움켜쥐었다. 그녀의 눈은 날카롭고 경계심이 가득했다.

그림자 속에서 카엘이 나타났다. 서른 살의 전직 기사는 검은 갑옷을 입고 있었고, 얼굴에는 오래된 흉터가 새겨져 있었다. 그의 회색 눈동자는 감정을 드러내지 않았다."""

def create_base_state(story=SAMPLE_STORY, art_style="fantasy illustration"):
    return {
        "content": story, 
        "completed_agents": [], 
        "errors": [], 
        "messages": [],
        "art_style": art_style,
        "rendering_engine": "Unreal Engine 5",
    }

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Appearance Agent 직접 Import 및 실행

> **참고**: LangGraph 의존성 문제를 피하기 위해 appearance 모듈만 직접 import

In [3]:
# 직접 모듈 import (팀 패키지 전체가 아닌 개별 파일)
import importlib.util
spec = importlib.util.spec_from_file_location(
    "appearance", 
    os.path.join(project_root, "app/agents/extraction/character/appearance.py")
)
appearance_module = importlib.util.module_from_spec(spec)

# llm 모듈 먼저 로드 필요
from app.agents.llm import get_structured_llm

# 모듈 실행
spec.loader.exec_module(appearance_module)

# 함수 가져오기
appearance_extraction_node = appearance_module.appearance_extraction_node
print("✅ Appearance Agent 모듈 로드 성공")

✅ Appearance Agent 모듈 로드 성공


In [4]:
# 실행
async def test_appearance():
    print("👗 Appearance Agent 테스트 (Production Level)...")
    state = create_base_state()
    result = await appearance_extraction_node(state)
    return result

result = run_async(test_appearance())

# 에러 확인
if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    appearance_data = result.get('char_appearance', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(appearance_data)}개")
    print(f"   - 이름: {list(appearance_data.keys())}")

👗 Appearance Agent 테스트 (Production Level)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['아린', '카엘']


## 2. ⚠️ Null Fallback 검증

> **핵심**: `null` 대신 `"unspecified"` 또는 role-based default 사용

In [5]:
appearance_data = result.get('char_appearance', {})

if not appearance_data:
    print("❌ 캐릭터 데이터 없음 - 위의 에러 메시지 확인")
else:
    print("="*70)
    print("⚠️ Null Fallback 검증")
    print("="*70)

    for name, data in appearance_data.items():
        print(f"\n🧑 {name}")
        
        # Check null fields
        null_fields = []
        filled_fields = []
        
        for field in ['physique', 'skin_tone', 'eyes', 'nose', 'mouth', 'hair_style', 'hair_color', 'expression']:
            value = data.get(field)
            if value is None:
                null_fields.append(field)
            elif value == 'unspecified':
                filled_fields.append(f"{field}: unspecified (fallback)")
            else:
                filled_fields.append(f"{field}: {value}")
        
        for f in filled_fields[:5]:
            print(f"   ✅ {f}")
        if len(filled_fields) > 5:
            print(f"   ... ({len(filled_fields) - 5} more)")
        
        if null_fields:
            print(f"   ❌ NULL 필드 발견: {null_fields}")
        else:
            print(f"   ✅ NULL 필드 없음 - Fallback 적용됨!")

⚠️ Null Fallback 검증

🧑 아린
   ✅ physique: 젊은 여전사
   ✅ skin_tone: <UNKNOWN>
   ✅ eyes: 날카롭고 경계심이 가득한
   ✅ nose: <UNKNOWN>
   ✅ mouth: <UNKNOWN>
   ... (3 more)
   ✅ NULL 필드 없음 - Fallback 적용됨!

🧑 카엘
   ✅ physique: 서른 살의 전직 기사
   ✅ skin_tone: <UNKNOWN>
   ✅ eyes: 회색 눈동자
   ✅ nose: <UNKNOWN>
   ✅ mouth: <UNKNOWN>
   ... (3 more)
   ✅ NULL 필드 없음 - Fallback 적용됨!


## 3. 🎨 Color Normalization 검증

> 자연어 색상 → Hex Code + Category

In [6]:
if not appearance_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🎨 Color Normalization 검증")
    print("="*70)

    for name, data in appearance_data.items():
        print(f"\n🧑 {name}")
        
        # Hair color
        hair_norm = data.get('hair_color_normalized', {})
        if hair_norm:
            print(f"   머리색:")
            print(f"      description: {hair_norm.get('description')}")
            print(f"      hex_code: {hair_norm.get('hex_code')}")
            print(f"      category: {hair_norm.get('category')}")
        else:
            print("   ❌ hair_color_normalized 없음")
        
        # Eye color
        eye_norm = data.get('eye_color_normalized', {})
        if eye_norm:
            print(f"   눈색:")
            print(f"      description: {eye_norm.get('description')}")
            print(f"      hex_code: {eye_norm.get('hex_code')}")
            print(f"      category: {eye_norm.get('category')}")

🎨 Color Normalization 검증

🧑 아린
   머리색:
      description: 검은
      hex_code: #000000
      category: BLACK
   눈색:
      description: 날카롭고 경계심이 가득한
      hex_code: None
      category: OTHER

🧑 카엘
   머리색:
      description: <UNKNOWN>
      hex_code: None
      category: OTHER
   눈색:
      description: 회색 눈동자
      hex_code: #808080
      category: GRAY


## 4. 🖼️ Full Visual Prompt 검증

> Image AI (DALL-E 3, Stable Diffusion)에 바로 사용 가능한 프롬프트

In [7]:
if not appearance_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🖼️ Full Visual Prompt (Image AI 입력용)")
    print("="*70)

    for name, data in appearance_data.items():
        prompt = data.get('full_visual_prompt', '')
        
        print(f"\n🧑 {name}")
        if prompt:
            print(f"\n   \"{prompt}\"")
            print(f"\n   [길이: {len(prompt)} chars]")
        else:
            print("   ❌ full_visual_prompt 없음")

🖼️ Full Visual Prompt (Image AI 입력용)

🧑 아린

   "젊은 여전사, <UNKNOWN> skin, 검은 hair, 긴 검은 머리카락, 날카롭고 경계심이 가득한 eyes, 경계심이 가득한 expression, 은빛 검, fantasy illustration"

   [길이: 111 chars]

🧑 카엘

   "서른 살의 전직 기사, <UNKNOWN> skin, <UNKNOWN> hair, <UNKNOWN>, 회색 눈동자 eyes, 감정을 드러내지 않은 expression, 검은 갑옷, 오래된 흉터, fantasy illustration"

   [길이: 128 chars]


## 5. 🎭 Style Context 검증

> 화풍/장르 메타데이터

In [8]:
if not appearance_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🎭 Style Context (메타데이터)")
    print("="*70)

    for name, data in appearance_data.items():
        style = data.get('style_context', {})
        
        print(f"\n🧑 {name}")
        print(f"   art_style: {style.get('art_style', 'N/A')}")
        print(f"   rendering_engine: {style.get('rendering_engine', 'N/A')}")

🎭 Style Context (메타데이터)

🧑 아린
   art_style: fantasy illustration
   rendering_engine: Unreal Engine 5

🧑 카엘
   art_style: fantasy illustration
   rendering_engine: Unreal Engine 5


## 6. Full JSON 출력

In [9]:
print("="*70)
print("📄 Full JSON Output (Production Ready)")
print("="*70)
if appearance_data:
    print(json.dumps(appearance_data, ensure_ascii=False, indent=2))
else:
    print("{}")
    print("\n❌ 데이터 없음 - 위의 에러 메시지 확인")

📄 Full JSON Output (Production Ready)
{
  "아린": {
    "name": "아린",
    "physique": "젊은 여전사",
    "skin_tone": "<UNKNOWN>",
    "eyes": "날카롭고 경계심이 가득한",
    "nose": "<UNKNOWN>",
    "mouth": "<UNKNOWN>",
    "hair_style": "긴 검은 머리카락",
    "hair_color": "검은",
    "attire": [
      "은빛 검"
    ],
    "expression": "경계심이 가득한",
    "scars_tattoos": [],
    "cyberware": [],
    "hair_color_normalized": {
      "description": "검은",
      "hex_code": "#000000",
      "category": "BLACK"
    },
    "eye_color_normalized": {
      "description": "날카롭고 경계심이 가득한",
      "hex_code": null,
      "category": "OTHER"
    },
    "full_visual_prompt": "젊은 여전사, <UNKNOWN> skin, 검은 hair, 긴 검은 머리카락, 날카롭고 경계심이 가득한 eyes, 경계심이 가득한 expression, 은빛 검, fantasy illustration",
    "style_context": {
      "art_style": "fantasy illustration",
      "rendering_engine": "Unreal Engine 5"
    }
  },
  "카엘": {
    "name": "카엘",
    "physique": "서른 살의 전직 기사",
    "skin_tone": "<UNKNOWN>",
    "eyes": "회색 눈동자",
    "nose":

## 7. Production 체크리스트

In [10]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

checks = []

# 1. 캐릭터 존재
if len(appearance_data) >= 2:
    checks.append(("✅", "2+ characters extracted"))
elif len(appearance_data) == 1:
    checks.append(("⚠️", f"Only 1 character (expected 2)"))
else:
    checks.append(("❌", f"Only {len(appearance_data)} characters"))

if appearance_data:
    # 2. Null Fallback
    has_null = False
    for name, data in appearance_data.items():
        for field in ['physique', 'skin_tone', 'eyes', 'expression']:
            if data.get(field) is None:
                has_null = True
                break

    if not has_null:
        checks.append(("✅", "Null Fallback: No null values in key fields"))
    else:
        checks.append(("❌", "Null Fallback: Still has null values"))

    # 3. Color Normalization
    has_hex = False
    for name, data in appearance_data.items():
        hair_norm = data.get('hair_color_normalized', {})
        if hair_norm.get('hex_code'):
            has_hex = True
            break

    if has_hex:
        checks.append(("✅", "Color Normalization: hex_code generated"))
    else:
        checks.append(("⚠️", "Color Normalization: No hex_code (color not in map?)"))

    # 4. Visual Prompt
    has_prompt = False
    for name, data in appearance_data.items():
        if data.get('full_visual_prompt'):
            has_prompt = True
            break

    if has_prompt:
        checks.append(("✅", "Prompt Aggregation: full_visual_prompt generated"))
    else:
        checks.append(("❌", "Prompt Aggregation: Missing full_visual_prompt"))

    # 5. Style Context
    has_style = False
    for name, data in appearance_data.items():
        if data.get('style_context', {}).get('art_style'):
            has_style = True
            break

    if has_style:
        checks.append(("✅", "Style Context: art_style defined"))
    else:
        checks.append(("❌", "Style Context: Missing art_style"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
total = len(checks)
print(f"결과: {passed}/{total} checks passed")

✅ Production 체크리스트

✅ 2+ characters extracted
✅ Null Fallback: No null values in key fields
✅ Color Normalization: hex_code generated
✅ Prompt Aggregation: full_visual_prompt generated
✅ Style Context: art_style defined

결과: 5/5 checks passed


## 8. 디버그: 에러 확인

In [11]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"\nResult keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Messages: {result.get('messages', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")

🔍 디버그 정보

Result keys: dict_keys(['char_appearance', 'completed_agents', 'messages'])
Errors: []
Messages: [{'role': 'appearance_agent', 'content': 'Extracted 2 character appearances (production-ready)'}]
Completed agents: ['appearance']
